# Benchmarking the `Baseline` Pipeline — A New User's Guide

This notebook shows **every way** to benchmark the `Baseline` knowledge-graph pipeline in Polygraph, and explains each approach along the way.

Two families of benchmarking:

1. **Whole-pipeline benchmarking** — run the entire `Baseline` pipeline end-to-end on a corpus and collect overall KG-quality metrics (`BenchmarkRunner`).
2. **Stage-specific benchmarking** — test one stage at a time (dedup, chunking, extraction, resolution, quality, RAG) against a gold dataset (`Benchmark.<Stage>`).

### Prerequisites
- Polygraph installed (`uv sync`) and run from the repo root.
- spaCy `en_core_web_sm` (Baseline's default extractor).
- Optional but recommended: the embedding model `paraphrase-multilingual-MiniLM-L12-v2` and `en_core_web_lg` for semantic chunking/dedup and the `Semantic()` pipeline.

> **Design note:** the library *fails loudly* — there are no silent fallbacks. If a stage or pipeline breaks, the benchmark raises. That is intentional: you always know when something is wrong.

## Setup

Run this first. It puts `src` on the path (if Polygraph is not already installed), imports the benchmarking API, and prints the built-in help for both benchmarking families.

In [ ]:
# ── Setup: imports + built-in help ─────────────────────────────────
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from polygraph._shared.stage_config import PreprocessConfig, ResolutionConfig
from polygraph.benchmark_pipeline import Benchmark, BenchmarkResult, BenchmarkRunner
from polygraph.benchmark_pipeline.compare import compare_matrix, compare_variants
from polygraph.benchmark_pipeline.config import ExperimentConfig
from polygraph.pipelines import Baseline

# What stage benchmarks are available, and how to use them.
print(Benchmark.help())

In [ ]:
# ── Small helpers used by the demos ────────────────────────────────

GOLD_DIR = Path("benchmarks/data")


def make_corpus(path: Path = Path("output/tutorial_corpus.jsonl")) -> Path:
    """Write a tiny corpus of 3 'good' articles.

    Each article is >200 chars and >40 words so every document survives the
    pipeline's default quality filter.
    """
    docs = [
        "Marie Curie was a physicist and chemist who conducted pioneering research on radioactivity. "
        "She discovered the chemical elements radium and polonium in Paris, France. She won two Nobel "
        "prizes and remains the only person to win Nobel prizes in two different sciences. Her work on "
        "radioactivity was revolutionary and she is widely regarded as one of the greatest scientists of all time.",
        "Albert Einstein was a theoretical physicist who developed the theory of relativity. He was born "
        "in Ulm, Germany in 1879 and later emigrated to the United States. He won the Nobel Prize in "
        "Physics for his work on the photoelectric effect. He is best known for the mass energy "
        "equivalence formula and spent his later years at Princeton University working on unified field theory.",
        "Alan Turing was a British mathematician and computer scientist who laid the foundations of "
        "modern computing. He formalised the concept of the Turing machine and made pivotal contributions "
        "to code-breaking during the second world war. His work on artificial intelligence and computability "
        "shaped the field for decades and continues to influence computer science today.",
    ]
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(json.dumps({"text": d}) for d in docs) + "\n", encoding="utf-8")
    return path


def sample_gold(name: str, n: int) -> Path:
    """Return a path to the first *n* records of a gold JSONL.

    Keeping the demos small (hundreds of records instead of thousands) makes
    the notebook run quickly while staying faithful to the real gold format.
    """
    src = GOLD_DIR / name
    out = Path(f"output/tutorial_{name}")
    out.parent.mkdir(parents=True, exist_ok=True)
    with src.open(encoding="utf-8") as f:
        lines = [next(f) for _ in range(n)]
    out.write_text("".join(lines), encoding="utf-8")
    return out


def show(title: str, result: BenchmarkResult) -> None:
    """Pretty-print a whole-pipeline ``BenchmarkResult``.

    Whole-pipeline metrics report graph size as ``num_nodes`` / ``num_edges`` /
    ``num_triples`` (the graph mixes entity, document, and chunk nodes).
    """
    print(f"── {title} ──")
    print(f"  overall_score : {result.overall_score:.3f}")
    print(f"  nodes / edges : {result.metrics.get('num_nodes')} / {result.metrics.get('num_edges')}")
    print(f"  triples       : {result.num_triples}")
    print(f"  pipeline      : {result.pipeline_variant}")
    print(f"  report        : {result.output_dir}/results_summary.json")

## Part 1 — Whole-pipeline benchmarking (`BenchmarkRunner`)

`BenchmarkRunner` runs the entire pipeline (preprocess → build_kg → evaluate → export) and returns a `BenchmarkResult`. It writes three artifacts to the output directory:

- `metrics.json` — quality metrics (`overall_score`, `completeness`, `consistency`, …)
- `results_summary.json` — the full serialized `BenchmarkResult`
- `knowledge_graph.json` — the generated KG

### 1a. Direct (programmatic)

Construct a runner with a pipeline *class*, input paths, and an output dir, then call `run()`. Here we use sentence chunking + minhash dedup so the demo is fast and needs no embedding model.

In [ ]:
# ── 1a. Whole-pipeline, direct mode ────────────────────────────────
corpus = make_corpus()

runner = BenchmarkRunner(
    pipeline=Baseline,
    input_paths=[str(corpus)],
    output_dir="output/tutorial_whole_pipeline",
    extra={
        "preprocess": PreprocessConfig(
            chunk_method="sentence", doc_dedup_method="minhash", chunk_dedup_method="minhash"
        )
    },
)
result = runner.run()
show("Whole-pipeline Baseline (direct)", result)

### 1b. From a YAML config (reproducible)

`ExperimentConfig.from_yaml` parses a config describing the whole run — pipeline variant, stage settings, input, output — and `BenchmarkRunner.from_config` executes it. See `experiments/kg/001_baseline/config.yaml` for a real example.

> Note: configs that don't override `preprocess` use the defaults — semantic chunking + layered dedup — so this cell needs the embedding model cached.

In [ ]:
# ── 1b. Whole-pipeline, from a YAML config ─────────────────────────
import yaml

config_yaml = {
    "name": "tutorial — baseline on a tiny corpus",
    "description": "Whole-pipeline benchmark driven by a YAML config.",
    "pipeline": {
        "variant": "baseline",
        "extraction": {
            "mode": "composed",
            "entity_method": "spacy",
            "relation_method": "ontology_rules",
        },
        "resolution": {"method": "string", "threshold": 0.85},
        "build": {"method": "networkx"},
    },
    "input": {"paths": [str(corpus)]},
    "output": {"dir": "output/tutorial_yaml"},
}
config_path = Path("output/tutorial_config.yaml")
config_path.write_text(yaml.safe_dump(config_yaml), encoding="utf-8")

config = ExperimentConfig.from_yaml(config_path)
runner = BenchmarkRunner.from_config(config)
result = runner.run()
show("Whole-pipeline Baseline (YAML)", result)

### 1c. Comparing pipelines & sweeping configurations

- `compare_variants(...)` — run registered pipeline variants side-by-side (`"surface"` = Baseline, `"semantic"` = Semantic).
- `compare_matrix(...)` — run every combination of variant × resolution × chunker × dedup method.
- `BenchmarkRunner.compare(baseline, variant, ...)` — head-to-head of two pipeline classes.

In [ ]:
# ── 1c. Comparing variants & a parameter sweep ─────────────────────
results = compare_variants(
    variants=["surface"],  # "surface" is the registered name for Baseline
    input_paths=[str(corpus)],
    output_dir="output/tutorial_compare",
    extra={
        "preprocess": PreprocessConfig(
            chunk_method="sentence", doc_dedup_method="minhash", chunk_dedup_method="minhash"
        )
    },
)
for name, r in results.items():
    print(f"{name:8} score={r.overall_score:.3f} nodes={r.metrics.get('num_nodes')} triples={r.num_triples}")

# Full parameter sweep: every combination of variant × resolution × chunker × dedup.
# Pass chunk_dedup_methods explicitly — otherwise chunk dedup keeps its default
# ('layered', which needs the embedding model).
results = compare_matrix(
    variants=["surface"],
    input_paths=[str(corpus)],
    output_dir="output/tutorial_matrix",
    resolutions=["string"],
    chunkers=["sentence"],
    doc_dedup_methods=["minhash"],
    chunk_dedup_methods=["minhash"],
)
for combo, r in results.items():
    print(f"{combo:28} score={r.overall_score:.3f}")

## Part 2 — Stage-specific benchmarks (`Benchmark.<Stage>`)

Each stage benchmark tests **one stage** against a gold dataset, using whatever configuration the pipeline was built with. This isolates *where* a pipeline loses quality.

| Stage      | What it tests                        | Gold dataset                 | Primary metrics          |
|------------|--------------------------------------|------------------------------|--------------------------|
| Dedup      | Detecting near-duplicate documents   | DBLP-ACM (auto-downloaded)   | Precision / Recall / F1   |
| Chunking   | Chunks keep gold entities intact     | CoNLL-derived                | Boundary P / R / F1       |
| Extraction | NER spans vs gold entities           | CoNLL-2003                   | Span P / R / F1           |
| Resolution | Merging name variants into entities  | T2D (placeholders)           | Cluster / pairwise F1     |
| Quality    | Keep vs reject low-quality text      | TACRED (placeholder)         | Accuracy / P / R / F1     |
| RAG        | KG retrieval for QA                  | HotpotQA (supply your own)   | Entity recall@k, …        |

### Dedup & Chunking

**Dedup** scores the pipeline's document-dedup decisions against gold duplicate/non-duplicate pairs (DBLP-ACM — auto-downloaded). **Chunking** checks that the boundaries the chunker produces don't split gold entities (boundary F1).

In [ ]:
# ── Dedup: duplicate detection vs gold pairs ───────────────────────
dedup_result = Benchmark.Dedup().run(
    pipelines={
        "exact": Baseline(preprocess=PreprocessConfig(doc_dedup_method="exact")),
        "minhash": Baseline(preprocess=PreprocessConfig(doc_dedup_method="minhash")),
    }
)
for name, m in dedup_result.items():
    print(f"Dedup {name:8} P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f} ({m['method']})")

# ── Chunking: do chunks keep gold entities intact? ─────────────────
chunk_gold = sample_gold("chunking_gold.jsonl", 200)
chunk_result = Benchmark.Chunking(dataset=chunk_gold).run(
    pipelines={
        "sentence": Baseline(preprocess=PreprocessConfig(chunk_method="sentence")),
        "fixed": Baseline(preprocess=PreprocessConfig(chunk_method="fixed")),
    }
)
for name, m in chunk_result.items():
    print(f"Chunk  {name:8} P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f} ({m['method']})")

### Extraction & Resolution

**Extraction** compares the extractor's entity spans against gold NER annotations (span-level precision/recall/F1). **Resolution** checks how well the pipeline merges name variants ("IBM", "International Business Machines") into one entity (cluster F1). The bundled gold is a small placeholder — use `dataset=` / `--dataset` to supply your own.

In [ ]:
# ── Extraction: NER quality vs gold spans ──────────────────────────
ext_gold = sample_gold("ner_gold.jsonl", 100)
ext_result = Benchmark.Extraction(dataset=ext_gold).run(pipelines={"surface": Baseline()})
m = ext_result["surface"]
print(
    f"Extraction P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f} "
    f"type_acc={m['type_accuracy']:.3f} ({m['entity_method']})"
)

# ── Resolution: merging name variants ──────────────────────────────
res_result = Benchmark.Resolution().run(
    pipelines={
        "string": Baseline(),
        "embed": Baseline(resolution=ResolutionConfig(method="embedding")),
    }
)
for name, m in res_result.items():
    print(
        f"Resolution {name:6} P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f} "
        f"(pairwise F1={m.get('pairwise_f1')})"
    )

### Quality & RAG

**Quality** scores the quality filter against keep/reject labels. Real TACRED data is license-gated, so a clearly-labeled synthetic placeholder is auto-generated; replace it for real numbers. **RAG** builds a KG from input documents and measures retrieval against gold QA pairs (entity recall@k, chunk retrieval). The canonical HotpotQA host is often offline, so this demo uses a tiny self-contained gold — supply your own via `dataset=` for real measurements.

In [ ]:
# ── Quality: keep/reject accuracy ──────────────────────────────────
quality_result = Benchmark.Quality().run(pipelines={"surface": Baseline()})
m = quality_result["surface"]
print(f"Quality acc={m['accuracy']:.3f} P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f}")

# ── RAG: retrieval from a KG built by the pipeline ─────────────────
rag_gold = Path("output/tutorial_rag_gold.jsonl")
rag_gold.write_text(
    "\n".join(
        [
            json.dumps(
                {
                    "query": "Who discovered radium?",
                    "answer_entity": "Marie Curie",
                    "supporting_chunks": ["She discovered the chemical elements radium and polonium in Paris, France."],
                }
            ),
            json.dumps(
                {
                    "query": "Where was Einstein born?",
                    "answer_entity": "Ulm",
                    "supporting_chunks": ["He was born in Ulm, Germany in 1879."],
                }
            ),
        ]
    )
    + "\n",
    encoding="utf-8",
)
rag_result = Benchmark.RAG(dataset=rag_gold).run(
    pipelines={
        "baseline": Baseline(
            preprocess=PreprocessConfig(
                chunk_method="sentence", doc_dedup_method="minhash", chunk_dedup_method="minhash"
            )
        )
    },
    input_paths=[str(corpus)],
    k=3,
)
m = rag_result["baseline"]
print(
    f"RAG  entity_recall@3={m['entity_recall_at_k']:.3f} entity_coverage={m['entity_coverage']:.3f} "
    f"chunk_recall@3={m['chunk_recall_at_k']:.3f}"
)

## Reading the results

A `BenchmarkResult` gives convenience accessors plus the full detail:

- `overall_score`, `num_entities`, `num_triples` — quick quality headline
- `metrics` — the raw quality metrics dict (from `metrics.json`)
- `structural_audit` — graph-structure diagnostics
- `artifacts` — paths to generated files (`.json`, `.graphml`)
- `config_snapshot` — the experiment config, for reproducibility
- `output_dir` — where `metrics.json` and `results_summary.json` were written

In [ ]:
# ── Inspecting a whole-pipeline result and its report ──────────────
r = BenchmarkRunner(
    pipeline=Baseline,
    input_paths=[str(corpus)],
    output_dir="output/tutorial_report",
    extra={
        "preprocess": PreprocessConfig(
            chunk_method="sentence", doc_dedup_method="minhash", chunk_dedup_method="minhash"
        )
    },
).run()

headline = {
    "overall_score": r.overall_score,
    "nodes / edges": r.metrics.get("num_nodes"),
    "triples": r.num_triples,
    "completeness": r.metrics.get("completeness"),
    "consistency": r.metrics.get("consistency"),
}
print("headline metrics:", json.dumps(headline, indent=2))
print("artifacts        :", r.artifacts)
print("summary written  :", (Path(r.output_dir) / "results_summary.json").exists())
print("output dir       :", r.output_dir)

## Interpreting scores & tips

- **Stage vs whole-pipeline** — stage benchmarks isolate ONE stage against gold data; the whole-pipeline benchmark measures the end product. Use stage scores to find *where* quality is lost, and the overall score to compare pipelines.
- **Scores are honest.** Low numbers are findings, not bugs. For example, DBLP duplicate pairs differ only by casing, so case-sensitive methods (`exact`, `minhash`) underperform — a real signal that normalization matters.
- **No silent fallbacks.** If a pipeline breaks, the benchmark raises. When something returns zero or crashes, fix the cause rather than hiding it.
- **Models.** Semantic chunking/dedup and embedding resolution need the embedding model; `Semantic()` also needs `en_core_web_lg`. The demos above use `sentence`/`minhash` so they run without them.
- **Gold data.** `dedup`, `chunking`, `extraction`, `resolution` ship or auto-download gold; `rag` (HotpotQA) and real `quality` (TACRED) gold must be supplied by you via `dataset=` / `--dataset`.

### Measuring variance across runs

Whole-pipeline quality scores are deterministic for fixed inputs, but *timing* varies run to run. `timeit` is handy for stable timing of a small run:

In [ ]:
# ── Stable timing across repetitions ───────────────────────────────
import timeit


def run_once() -> None:
    BenchmarkRunner(
        pipeline=Baseline,
        input_paths=["output/tutorial_corpus.jsonl"],
        output_dir="output/tutorial_timing",
        extra={
            "preprocess": PreprocessConfig(
                chunk_method="sentence", doc_dedup_method="minhash", chunk_dedup_method="minhash"
            )
        },
    ).run()


times = timeit.repeat(run_once, number=1, repeat=3)
print("whole-pipeline run times (s):", [round(t, 3) for t in times])
print(f"mean = {sum(times) / len(times):.3f}s   min = {min(times):.3f}s")

## Summary

You now know how to benchmark the `Baseline` pipeline in every way the library supports:

1. **Whole-pipeline** — `BenchmarkRunner` (direct or YAML), `BenchmarkRunner.compare`, `compare_variants`, `compare_matrix`.
2. **Stage-specific** — `Benchmark.Dedup` / `Chunking` / `Extraction` / `Resolution` / `Quality` / `RAG` against gold data.
3. **Results** — `BenchmarkResult`, `metrics.json`, `results_summary.json`.
4. **Variance** — `timeit` across repetitions for stable timing.

To benchmark every stage at once from the command line:

```bash
uv run python tools/run_benchmarks.py --stage dedup chunking extraction resolution quality
```

Now try swapping `Baseline` for `Semantic()`, or a custom `Pipeline` subclass, and compare the results!